# Challenge Three — Testing & Evaluation
**Author:** Aaron

Two Gemini-backed functions for the Aurora Bay civic assistant, `pytest` unit tests for each, and a
Gen AI Evaluation Service comparison of two prompt styles.

## Requirement -> implementation
| # | Requirement | Where |
|---|---|---|
| 2 | Gemini function classifying questions (Employment / General Information / Emergency Services / Tax Related) | `classify_question()` |
| 3 | Gemini function generating government social-media posts | `generate_social_post()` |
| 4 | Unit tests for each function with pytest | `ipytest` cell |
| 5 | Evaluate & compare responses from different prompts (Evaluation API) | `EvalTask` comparison |


## 1. Install & configure

In [10]:
%pip install --quiet --upgrade google-genai "google-cloud-aiplatform[evaluation]" ipytest pandas

In [11]:
import os, google.auth
try:
    _c, _p = google.auth.default()
except Exception:
    _p = None
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or _p
assert PROJECT_ID, "Set GOOGLE_CLOUD_PROJECT."

GENAI_LOCATION = "global"        # Gemini endpoint
EVAL_LOCATION  = "us-east1"   # Gen AI Evaluation Service / Vertex Experiments region
print("Project:", PROJECT_ID)

Project: qwiklabs-gcp-02-d49a612b5dc6


In [12]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location=GENAI_LOCATION)

def resolve_model(cands):
    for m in cands:
        try:
            client.models.generate_content(model=m, contents="ping",
                config=types.GenerateContentConfig(max_output_tokens=8))
            return m
        except Exception as e:
            print(f"  {m} unavailable: {type(e).__name__}")
    raise RuntimeError("No candidate Gemini model available.")

MODEL = resolve_model(["gemini-3.1-flash", "gemini-2.5-flash", "gemini-2.0-flash"])
print("Using model:", MODEL)

  gemini-3.1-flash unavailable: ClientError
Using model: gemini-2.5-flash


## Requirement 2 — classify a citizen question

Returns exactly one of the four categories. Temperature 0 for determinism, plus a normalization
pass so the output always lands on a valid label.

In [13]:
CATEGORIES = ["Employment", "General Information", "Emergency Services", "Tax Related"]

def classify_question(question: str) -> str:
    instruction = (
        "Classify the citizen's question into exactly ONE of these categories: "
        + ", ".join(CATEGORIES) + ". Reply with only the category name."
    )
    resp = client.models.generate_content(
        model=MODEL, contents=f"{instruction}\n\nQuestion: {question}",
        config=types.GenerateContentConfig(temperature=0))
    out = (resp.text or "").strip()
    for c in CATEGORIES:               # normalize to a known label
        if c.lower() in out.lower():
            return c
    return out

## Requirement 3 — generate a government social-media post

For announcements like weather emergencies, holidays, or school closings.

In [14]:
def generate_social_post(announcement: str) -> str:
    instruction = (
        "Write a concise, friendly social-media post for an official town announcement. "
        "Keep it under 280 characters and include a clear call to action.")
    resp = client.models.generate_content(
        model=MODEL, contents=f"{instruction}\n\nAnnouncement: {announcement}",
        config=types.GenerateContentConfig(temperature=0.4))
    return (resp.text or "").strip()

In [15]:
# Quick look at both functions.
print(classify_question("How do I apply for a job with the town?"))
print(classify_question("There's a gas leak on Main Street, who do I call?"))
print(generate_social_post("All Aurora Bay schools are closed today due to heavy snow."))

Employment
Emergency Services
Official Announcement: All Aurora Bay schools are CLOSED today due to heavy snow. Please stay safe & warm! Visit [YourTownWebsite.org] for updates.


## Requirement 4 — unit tests with pytest

`ipytest` runs pytest inside the notebook so the results are saved with the cell output. Tests use
clear-cut inputs and assert on the normalized category / topic keywords (robust to Gemini's
phrasing).

In [16]:
import ipytest
ipytest.autoconfig()

In [17]:
%%ipytest

def test_classify_returns_valid_category():
    assert classify_question("What are the library hours?") in CATEGORIES

def test_classify_employment():
    assert classify_question("How do I apply for a job with the town?") == "Employment"

def test_classify_emergency():
    assert classify_question("There is a fire at the harbor, who do I call?") == "Emergency Services"

def test_social_post_nonempty():
    post = generate_social_post("Town hall closed Monday for the holiday.")
    assert isinstance(post, str) and len(post.strip()) > 0

def test_social_post_mentions_topic():
    post = generate_social_post("All schools are closed today due to a snowstorm.")
    assert "school" in post.lower()

.....                                                                                        [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
5 passed, 1 warning in 15.43s


## Requirement 5 — evaluate & compare two prompts (Evaluation API)

Generate social posts for the same announcements under two prompt styles — a plain one and a more
engaging one — then score both with the Gen AI Evaluation Service and compare the summary metrics.
This is the `EvalTask` flow: build a dataset of `prompt`/`response`, pick metrics, call `evaluate()`.

In [18]:
import pandas as pd
import vertexai
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

vertexai.init(project=PROJECT_ID, location=EVAL_LOCATION)

announcements = [
    "City offices closed Monday for Indigenous Peoples' Day.",
    "Boil-water advisory in effect for the harbor district until further notice.",
    "Free flu shots available at the community center this Saturday.",
    "Heavy snow expected tonight; plows will run on emergency routes first.",
]

PROMPTS = {
    "plain":    "Write a social media post for this announcement: {a}",
    "engaging": ("Write an engaging, friendly social media post for this town announcement. "
                 "Include a clear call to action and one relevant emoji: {a}"),
}

METRICS = [
    MetricPromptTemplateExamples.Pointwise.FLUENCY,
    MetricPromptTemplateExamples.Pointwise.COHERENCE,
    MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
]

def gen(tmpl, a):
    return client.models.generate_content(
        model=MODEL, contents=tmpl.format(a=a),
        config=types.GenerateContentConfig(temperature=0.4)).text

summary = {}
for name, tmpl in PROMPTS.items():
    df = pd.DataFrame({
        "prompt":   [tmpl.format(a=a) for a in announcements],
        "response": [gen(tmpl, a) for a in announcements],
    })
    task = EvalTask(dataset=df, metrics=METRICS, experiment="challenge3-social-posts")
    result = task.evaluate(experiment_run_name=f"prompt-{name}")
    summary[name] = result.summary_metrics

import json
print(json.dumps(summary, indent=2, default=str))

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:13<00:00,  1.11s/it]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:13.352587844001391 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:14<00:00,  1.18s/it]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:14.170734119999906 seconds


{
  "plain": {
    "row_count": 4,
    "fluency/mean": 5.0,
    "fluency/std": 0.0,
    "coherence/mean": 5.0,
    "coherence/std": 0.0,
    "instruction_following/mean": 5.0,
    "instruction_following/std": 0.0
  },
  "engaging": {
    "row_count": 4,
    "fluency/mean": 5.0,
    "fluency/std": 0.0,
    "coherence/mean": 5.0,
    "coherence/std": 0.0,
    "instruction_following/mean": 4.75,
    "instruction_following/std": 0.5
  }
}


## Submission notes
- Run all top-to-bottom so the pytest results and eval summary are saved, then commit to `challenge3/`.
- The Evaluation step uses an LLM autorater and logs a run to Vertex AI Experiments; it needs the
  Vertex AI API enabled and runs in `us-east1`.
- Compare the two prompts' summary metrics in the final output to show which prompt style scores higher.
